## Ensemble Learning

- Ensemble learning : the process of building a complex machine learning model by combining multiple base estimators as building blocks. 

### Base Estimators:

- Tend to underachieve on their own: 
    -  weak learners : considered to have high bias or high variance.
    - The nature of weakness of the base model is a design choice when determining the best ensembling method. 

- Basic components of esembling learning:

    - Base models that are weak
    - An ensembling method that combines the base models to improve proformance and robustness

- Base estimators should be uncorrelated and independent
    - having a higher diversity among the trained base estimators leads to a stronger ensemble model.

- Common ensembling mothods :

- **Bagging:** 
    - Boostrape Aggregating : is an esemble method that combines he concepts of bootstrapping and aggregation. 
    - Can be used on both classification and regression problems.
    - use weak learners as base models that are complex and tend to suffer from high variance.
    - Their weakness as models is due to being built with only a subset of the available features and on a subset of the training data due to bootstrapping.
    - Bootstrapping refers to the method of sampling data with replacement.
    - Each of the bse models is trained independetly of the others. 
    - Each base model is trained using only a subset of the original features. 
    - base models overfit to both a subset of the available training data and a subset of the available features. 
    - decision trees that are large and overfit to the bootstrapped subset of data provided to each of them.
    - aggregated : majority vote for classification problems, and averaging for regression problems.

- **Boosting:**
    - the weak learners are too simple and tend to suffer from high bias.
    - the base models are decision trees with only one level, a decision stump. 
    - Decision stumps can only make a decision based off of one feature at a time, causing them to underfit the data substantially.
    - A sequential learning technique where each of the base models builds off the previous model. 
    - aims to imporove performance of the ensembled model by atempting to fix the errors in the previous stage.
    - Common implementations of the boosting algorithm are:
        - Adaptive Boosting
        - Gradient Boosting


![bagging and boosting](images/bagging_boosting.png)

- **Stacking:**
    - Extremely flexible ensembling technique where the final model is trained to learn how to best combine a set of base moels to make strong predictions.
    - Base models do not need to be nsame type of learning algorithm.
    - Can be used to combine weak and decent performing learners.

## Random Forest:

- Decision trees : 
    - prone to overfitting

- Random Forest: an ensemble machine learning technique that contains many decision trees that all work together to classify new points.
    - when asked to classify a new point, the random forest gives that point to each of the decision trees. 
    - each report their classification and the random forest returns the most popular classifacation. 
    > Some of the trees in the random forest may be overfit, but by making the prediction based on a large number of trees, overfitting will have less of an impact.

- **Random forest is made with the bagging technique.**

### Bootstrapping:
![bootstrapping example](images/bootstrapping_example.png)

---


## Bootstrapping: 

```python
import pandas as pd
import numpy as np
import codecademylib3
import matplotlib.pyplot as plt
import seaborn as sns

#Models from scikit learn module:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data', names=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'accep'])
df['accep'] = ~(df['accep']=='unacc') #1 is acceptable, 0 if not acceptable
X = pd.get_dummies(df.iloc[:,0:6], drop_first=True)
y = df['accep']

x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)
nrows = df.shape[0]

## 1. Print number of rows and distribution of safety ratings
print(nrows)
print(f'Distribution of safety ratings in {nrows} of data:')
print(df.safety.value_counts(normalize=True))

## 2. Create bootstrapped sample
boot_sample = df.sample(nrows, replace=True)
print(f'Distribution of safety ratings in bootstrapped sample data: {boot_sample.safety.value_counts(normalize=True)}')

## 3. Create 1000 bootstrapped samples
low_perc = []
for _ in range(1000):
  sample = df.sample(nrows, replace=True)
  low_percentage = (sample['safety'] == 'low').mean()
  low_perc.append(low_percentage) 

## 4. Plot a histogram of the low percentage values
mean_lp = np.mean(low_perc) 
print(mean_lp)
plt.hist(low_perc, bins=20);
plt.xlabel('Low Percentage')
plt.show()

## 5. What are the 2.5 and 97.5 percentiles?
print(f'Average low percentage: {np.mean(low_perc).round(4)}')

low_perc.sort()
print(f'95% Confidence Interval for low percengage: ({low_perc[25].round(4)},{low_perc[975].round(4)})')
```

---

## Bagging: Decision Tree Classifier

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.metrics import accuracy_score

df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data', names=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'accep'])
df['accep'] = ~(df['accep']=='unacc') #1 is acceptable, 0 if not acceptable
X = pd.get_dummies(df.iloc[:,0:6], drop_first=True)
y = df['accep']
x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)

#1.Decision tree trained on  training set
dt = DecisionTreeClassifier(max_depth=5)
dt.fit(x_train, y_train)

print(f'Accuracy score of DT on test set (trained using full set): {dt.score(x_test, y_test).round(4)}')

#2. New decision tree trained on bootstrapped sample
dt2 = DecisionTreeClassifier(max_depth=5)
#ids are the indices of the bootstrapped sample
ids = x_train.sample(x_train.shape[0], replace=True, random_state=0).index
dt2.fit(x_train.loc[ids], y_train.loc[ids])
print(f'Accuracy score of DT on test set (trained using bootstrapped sample): {dt2.score(x_test, y_test).round(4)}')

## 3. Bootstapping ten samples and aggregating the results:
preds = []
random_state = 0
#Write for loop:
for i in range(10):
#ids are the indices of the bootstrapped sample
  y_pred = DecisionTreeClassifier(max_depth=5)
  ids = x_train.sample(x_train.shape[0], replace=True, random_state=random_state+i).index
  y_pred.fit(x_train.loc[ids], y_train.loc[ids])
  preds.append(y_pred.predict(x_test))

ba_pred = np.array(preds).mean(0)

# 4. Calculate accuracy of the bagged sample
ba_accuracy = accuracy_score(ba_pred>=0.5, y_test)
print(f'Accuracy score of aggregated 10 bootstrapped samples:{ba_accuracy.round(4)}')
```

---

## Random Feature Selection:

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data', names=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'accep'])
df['accep'] = ~(df['accep']=='unacc') #1 is acceptable, 0 if not acceptable
X = pd.get_dummies(df.iloc[:,0:6], drop_first=True)
y = df['accep']
x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)
dt = DecisionTreeClassifier()
dt.fit(x_train, y_train)
print("Accuracy score of DT on test set (trained using full feature set):")
accuracy_dt = dt.score(x_test, y_test)
print(accuracy_dt.round(4))

# 1. Create rand_features, random samples from the set of features
rand_features = np.random.choice(x_train.columns,10)

# Make new decision tree trained on random sample of 10 features and calculate the new accuracy score
dt2 = DecisionTreeClassifier()
dt2.fit(x_train[rand_features], y_train)
accuracy_dt2 = dt2.score(x_test[rand_features], y_test)
print("Accuracy score of DT on test set (trained using random feature sample):", accuracy_dt2.round(4))

# 2. Build decision trees on 10 different random samples 
predictions = []
for i in range(10):
    rand_features = np.random.choice(x_train.columns,10)
    dt2.fit(x_train[rand_features], y_train)
    predictions.append(dt2.predict(x_test[rand_features]))

## 3. Get aggregate predictions and accuracy score
prob_predictions = np.array(predictions).mean(0)
agg_predictions = (prob_predictions>0.5)
agg_accuracy = accuracy_score(agg_predictions, y_test)
print('Accuracy score of aggregated 10 samples:', agg_accuracy.round(4))
```

---

## Bagging : `BaggingClassifier()` Implementation

1. Instantiate am instance of `BaggingClassifier()`
2. Specify parameters:
    - `base_estimator` : refers to the machine learning model that is being bagged. (*in the case of the random forest, the base estimator would be a decision tree*)

```python
base_estimator = BaggingClassifier(DecisionTreeClassifier(max_depth=5))
```
```python
# After model has been defined, methods: 
# .fit()
# .predict()
# .score() can be used as expected.
```
**Hyperparameters specific to bagging:**

3. `n_estimators` : number of estimators desired to be used
4. `max_features` : maximum number of features to keep

> this procedure can be used for any based classifier or regression model

### Scikit-Learn Implementation:

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score

df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data', names=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'accep'])
df['accep'] = ~(df['accep']=='unacc') #1 is acceptable, 0 if not acceptable
X = pd.get_dummies(df.iloc[:,0:6], drop_first=True)
y = df['accep']
x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)

# 1. Bagging classifier with 10 Decision Tree base estimators
bag_dt = BaggingClassifier(DecisionTreeClassifier(max_depth=5), n_estimators=10)
bag_dt.fit(x_train, y_train)
bag_accuracy = accuracy_score(bag_dt.predict(x_test), y_test)
print('Accuracy score of Bagged Classifier, 10 estimators:', bag_accuracy.round(4))

# 2.Set `max_features` to 10.
bag_dt_10 = BaggingClassifier(DecisionTreeClassifier(max_depth=5), n_estimators=10, max_features=10)
bag_dt_10.fit(x_train, y_train)
bag_accuracy_10 = accuracy_score(bag_dt_10.predict(x_test), y_test)
print('Accuracy score of Bagged Classifier, 10 estimators, 10 max features:', bag_accuracy_10.round(4))

# 3. Change base estimator to Logistic Regression
from sklearn.linear_model import LogisticRegression
bag_lr = BaggingClassifier(LogisticRegression(),n_estimators=10, max_features=10)
bag_lr.fit(x_train, y_train)
bag_accuracy_lr = accuracy_score(bag_lr.predict(x_test), y_test)
print('Accuracy score of Logistic Regression, 10 estimators:', bag_accuracy_lr.round(4))
```

---

### Random Forest:

- Each split chooses a different random set

![Random forest example](images/random_forest_example.png)

> **Select as many features as the square root of the total number of features**

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data', names=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'accep'])
df['accep'] = ~(df['accep']=='unacc') #1 is acceptable, 0 if not acceptable
X = pd.get_dummies(df.iloc[:,0:6], drop_first=True)
y = df['accep']
x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)

# 1. Create a Random Forest Classifier and print its parameters
rf = RandomForestClassifier()
rf_params = rf.get_params()
print('Random Forest parameters:')
print(rf_params)

# 2. Fit the Random Forest Classifier to training data and calculate accuracy score on the test data
rf.fit(x_train, y_train)
y_pred = rf.predict(x_test)
rf_accuracy = rf.score(x_test, y_test)
print('Test set accuracy:')
print(rf_accuracy.round(2))

# 3. Calculate Precision and Recall scores and the Confusion Matrix
rf_precision = precision_score(y_test, y_pred)
print(f'Test set precision: {rf_precision.round(2)}')

rf_recall = recall_score(y_test, y_pred)
print(f'Test set recall: {rf_recall.round(2)}')

rf_confusion_matrix = confusion_matrix(y_test, y_pred)
print(f'Test set confusion matrix:\n{rf_confusion_matrix}')
```
---

### Random Forest Regressor:

```python
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data', names=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'accep'])
df['accep'] = ~(df['accep']=='unacc') #1 is acceptable, 0 if not acceptable
X = pd.get_dummies(df.iloc[:,0:6], drop_first=True)

## Generating some fake prices for regression! :) 
fake_prices = (15000 + 25*df.index.values)+np.random.normal(size=df.shape[0])*5000
df['price'] = fake_prices
print(df.price.describe())
y = df['price']

x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)

# 1. Create a Random Regressor and print `R^2` scores on training and test data
rfr = RandomForestRegressor()
rfr.fit(x_train, y_train)
r_squared_train = r2_score(y_train, rfr.predict(x_train))
r_squared_test = r2_score(y_test, rfr.predict(x_test))
print(f'Train set R^2: {r_squared_train.round(4)}')
print(f'Test set R^2: {r_squared_test.round(4)}')

# 2. Print Mean Absolute Error on training and test data
avg_price = np.mean(y)
print(f'Avg Price Train/Test: {avg_price}')

mae_train = mean_absolute_error(y_train, rfr.predict(x_train))
print(f'Train set MAE: {mae_train}')

mae_test = mean_absolute_error(y_test, rfr.predict(x_test))
print(f'Test set MAE: {mae_test}')
```

---

- Feature Importance : `feature_importance_` returns the more importance features of training set.